# 08 · Retratos recortados en círculo

Power BI no recorta imágenes, así que los retratos tienen que llegar ya redondos. Este cuaderno descarga el
retrato de Transfermarkt de cada jugador emparejado, lo recorta en un círculo centrado en la cara y lo guarda
en `images/jugadores/<id_jugador>.png`, la dirección a la que apunta `dim_jugador.url_foto`.

**Fuente: Transfermarkt.** Se usa la versión `big` del retrato (300 × 390 px), cambiando `medium` por `big` en
la url de `data/fotos_transfermarkt.csv`. Los originales se guardan en `data/bruto/retratos_tm/`, que no se
versiona; si ya están, no se vuelven a pedir.

**Encuadre** (aprobado por el autor sobre una muestra de 20 el 17/11/2026):

- Se detecta la cara con los clasificadores de OpenCV (frontal; si no, frontal alternativo; si no, perfil).
  Una cara más estrecha que el 45 % del ancho de la foto no se acepta: en estos retratos la cara ocupa casi
  todo el ancho, y una caja pequeña es un trozo de cara (un ojo, la boca). Sin cara válida, se usa la posición
  típica de un retrato de Transfermarkt.
- La cabeza se estima a partir de la caja de la cara: coronilla 0,40 alturas por encima y barbilla 1,25 por
  debajo de su borde superior. Se centra en vertical, un poco por debajo del centro del círculo.
- El círculo mide un 30 % más que el ancho de la foto, para que entre la cabeza entera: los retratos de
  Transfermarkt vienen tan cerrados que, del ancho de la foto, se cortaría la coronilla o la barbilla. En
  vertical el cuadrado nunca se sale de la foto: con retratos de 300 × 390, coge el alto completo.
- **Lo que queda fuera de la foto se rellena prolongando su borde y desenfocándolo mucho** (nunca con la cara).
  La foto nítida va encima con el borde fundido en 14 px. En fondos lisos no se nota; en fondos de estadio se ve
  una franja suave a los lados. Es lo único del retrato que no viene de la foto original.
- A los PNG con fondo transparente se les pone detrás gris claro `#f0f0f0`, el fondo más habitual en
  Transfermarkt.
- Máscara circular con esquinas transparentes (canal alfa), dibujada a 4 veces el tamaño para que el borde
  quede suave, y reducción final a 200 × 200 px.

In [1]:
# Instalamos las librerías del cuaderno; OpenCV 4 trae los clasificadores de caras
%pip install -q pandas requests pillow "opencv-python-headless<5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos io para abrir imágenes desde memoria
import io
# Importamos time para espaciar las descargas
import time
# Importamos requests para descargar los retratos
import requests
# Importamos numpy y OpenCV para detectar la cara y prolongar el borde
import numpy as np
import cv2
# Importamos pandas para las tablas
import pandas as pd
# Importamos Pillow para recortar y dibujar
from PIL import Image, ImageDraw, ImageFilter, ImageFont

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# User-agent identificable
CABECERAS = {"User-Agent": "scouting-laliga-2526/1.0 (proyecto de datos)"}
# Carpeta de los originales descargados (no versionada)
CRUDOS = "data/bruto/retratos_tm"
# Carpeta de los retratos recortados (versionada)
DESTINO = "images/jugadores"
# Creamos las dos
os.makedirs(CRUDOS, exist_ok=True)
os.makedirs(DESTINO, exist_ok=True)
# Tamaño final y factor de dibujo ampliado
LADO, ESCALA = 200, 4
# Ancho del círculo respecto al ancho de la foto
ZOOM = 1.30
# Fondo para los PNG sin fondo
RELLENO_PNG = (240, 240, 240)
# Clasificadores de caras de OpenCV, por orden de preferencia
CASCADAS = [("frontal", "haarcascade_frontalface_default.xml"), ("frontal alt2", "haarcascade_frontalface_alt2.xml"),
            ("perfil", "haarcascade_profileface.xml")]
# Los cargamos
CASCADAS = [(n, cv2.CascadeClassifier(cv2.data.haarcascades + f)) for n, f in CASCADAS]

# Jugadores con retrato de Transfermarkt aplicado
fotos = pd.read_csv("data/fotos_transfermarkt.csv")
# Solo los aplicados
fotos = fotos[fotos["aplicado"]].copy()
# Url de la versión grande
fotos["url_big"] = fotos["url_retrato"].str.replace("/portrait/medium/", "/portrait/big/", regex=False)
# Informamos
print("retratos a recortar:", len(fotos))

retratos a recortar: 539


## 1. Descarga de los originales

In [3]:
# Recorremos los retratos
for f in fotos.itertuples():
    # Fichero local del original
    ruta = os.path.join(CRUDOS, f"{f.id_jugador}.img")
    # Si ya está, no se vuelve a pedir
    if os.path.exists(ruta):
        continue
    # Pedimos la versión grande
    r = requests.get(f.url_big, headers=CABECERAS, timeout=30)
    # Si no responde, usamos la mediana
    if r.status_code != 200:
        r = requests.get(f.url_retrato, headers=CABECERAS, timeout=30)
    # Si tampoco, paramos
    r.raise_for_status()
    # Guardamos el original tal cual
    open(ruta, "wb").write(r.content)
    # Pausa entre descargas
    time.sleep(0.3)
# Informamos
print("originales disponibles:", sum(os.path.exists(os.path.join(CRUDOS, f"{i}.img")) for i in fotos["id_jugador"]))

originales disponibles: 539


## 2. Recorte

In [4]:
# Detecta la cara más grande; devuelve (caja, método)
def detectar(im):
    # Escala de grises con el contraste ecualizado
    gris = cv2.equalizeHist(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY))
    # Tamaño mínimo de una cara válida: el 45 % del ancho de la foto
    minimo = int(0.45 * gris.shape[1])
    # Probamos cada clasificador, primero exigente y luego más permisivo
    for nombre, cascada in CASCADAS:
        for vecinos in (5, 3):
            # Caras detectadas
            caras = cascada.detectMultiScale(gris, scaleFactor=1.05, minNeighbors=vecinos, minSize=(minimo, minimo))
            # La más grande
            if len(caras):
                return max(caras, key=lambda c: c[2] * c[3]), nombre
    # Sin cara
    return None, "sin cara"


# Encuadra un retrato en un cuadrado centrado en la cabeza
def encuadrar(im):
    # Con canal alfa para detectar los PNG sin fondo
    rgba = im.convert("RGBA")
    # PNG sin fondo: gris claro detrás
    if rgba.getchannel("A").getextrema()[0] < 255:
        # Fondo gris
        base = Image.new("RGBA", rgba.size, RELLENO_PNG + (255,))
        # Retrato encima
        base.alpha_composite(rgba)
        rgba = base
    # Imagen en color sin alfa
    rgb = rgba.convert("RGB")
    # Tamaño del original
    W, H = rgb.size
    # Cara
    caja, metodo = detectar(rgb)
    # Sin cara: posición típica de un retrato de Transfermarkt
    if caja is None:
        caja = (W * 0.18, H * 0.20, W * 0.64, W * 0.64)
    # Caja de la cara
    x, y, w, h = caja
    # Coronilla y barbilla estimadas
    coronilla, barbilla = y - 0.40 * h, y + 1.25 * h
    # Lado del cuadrado
    lado = int(round(ZOOM * W))
    # Centro vertical: la cabeza un poco por debajo del centro
    cy = (coronilla + barbilla) / 2 + 0.06 * lado
    # Esquina superior izquierda del cuadrado sobre la foto
    arr, izq = int(round(cy - lado / 2)), int(round(W / 2 - lado / 2))
    # Nunca se rellena por arriba ni por abajo: el cuadrado no se sale de la foto en vertical
    # (con fotos de 300 × 390, el lado es 390 y el cuadrado coge el alto completo)
    arr = min(max(arr, min(0, H - lado)), max(0, H - lado))
    # Margen para prolongar el borde
    pad = max(0, -izq, izq + lado - W, -arr, arr + lado - H) + 30
    # Se prolonga el borde de la foto y se desenfoca mucho
    a = cv2.GaussianBlur(cv2.copyMakeBorder(np.array(rgb), pad, pad, pad, pad, cv2.BORDER_REPLICATE), (0, 0), 22)
    # Fondo del cuadrado
    lienzo = Image.fromarray(a[arr + pad: arr + pad + lado, izq + pad: izq + pad + lado])
    # Máscara de la foto nítida con el borde fundido
    mascara = Image.new("L", (W, H), 0)
    ImageDraw.Draw(mascara).rectangle([14, 14, W - 15, H - 15], fill=255)
    mascara = mascara.filter(ImageFilter.GaussianBlur(7))
    # Foto nítida encima
    lienzo.paste(rgb, (-izq, -arr), mascara)
    # Devolvemos el cuadrado y el método de detección
    return lienzo, metodo


# Recorta en círculo con esquinas transparentes
def circulo(im):
    # Ampliamos para suavizar el borde
    im = im.convert("RGBA").resize((LADO * ESCALA, LADO * ESCALA), Image.LANCZOS)
    # Máscara circular
    m = Image.new("L", im.size, 0)
    ImageDraw.Draw(m).ellipse([0, 0, im.width - 1, im.height - 1], fill=255)
    # Aplicamos la máscara
    im.putalpha(m)
    # Tamaño final
    return im.resize((LADO, LADO), Image.LANCZOS)


# Recortamos los 539
filas = []
for f in fotos.itertuples():
    # Original
    im = Image.open(os.path.join(CRUDOS, f"{f.id_jugador}.img"))
    # Tamaño original
    tam = im.size
    # Encuadre y círculo
    cuadro, metodo = encuadrar(im)
    # Ruta de destino
    ruta = os.path.join(DESTINO, f"{f.id_jugador}.png")
    # Guardamos
    circulo(cuadro).save(ruta, optimize=True)
    # Anotamos
    filas.append({"id_jugador": f.id_jugador, "jugador": f.jugador, "url_original": f.url_big,
                  "ancho_original": tam[0], "alto_original": tam[1], "deteccion": metodo,
                  "kb": round(os.path.getsize(ruta) / 1024, 1)})
# Tabla de control
recortes = pd.DataFrame(filas)
# Guardamos la trazabilidad de cada recorte
recortes.to_csv("data/retratos_recortados.csv", index=False)
# Resumen
print(recortes["deteccion"].value_counts().to_string())
print("tamaño original:", recortes.groupby(["ancho_original", "alto_original"]).size().to_dict())
print(f"peso: mínimo {recortes['kb'].min()} KB · máximo {recortes['kb'].max()} KB · total {recortes['kb'].sum() / 1024:.1f} MB")
# Ninguno pasa de 128 KB, el límite de Tableau
assert (recortes["kb"] < 128).all()

deteccion
frontal         511
frontal alt2     15
sin cara         10
perfil            3
tamaño original: {(300, 390): 539}
peso: mínimo 48.0 KB · máximo 79.7 KB · total 32.6 MB


## 3. Hoja de control

Las 539 juntas, sobre el fondo del panel, para revisarlas de un vistazo. No se versiona: contiene retratos de
Transfermarkt.

In [5]:
# Columnas de la hoja y tamaño de cada retrato en ella
COLS, T = 20, 64
# Filas necesarias
filas_hoja = -(-len(recortes) // COLS)
# Lienzo con el fondo del panel
hoja = Image.new("RGB", (COLS * (T + 6) + 10, filas_hoja * (T + 6) + 10), "#07090e")
# Recorremos los retratos
for n, f in enumerate(recortes.itertuples()):
    # Retrato reducido
    im = Image.open(os.path.join(DESTINO, f"{f.id_jugador}.png")).resize((T, T), Image.LANCZOS)
    # Lo pegamos respetando la transparencia
    hoja.paste(im, (10 + (n % COLS) * (T + 6), 10 + (n // COLS) * (T + 6)), im)
# Guardamos fuera de la carpeta de retratos
hoja.save("images/control_retratos.png")